<a href="https://colab.research.google.com/github/dipikamishra/my-repository/blob/main/03_DataSimilarityDistance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Similarity and Distance

In this week's practical we will learn and practice the following skills:
- Load and verify a data set
- Understanding the importance of data scaling for distance calculations
- Apply multiple different distance metrics to numerical data
- Compare categorical data with similarity measures
- Computing similarity for mixed numerical/categorical data
- Identifying nearest neighbours
- Computing an all-vs-all distace matrix
- Documenting our work



## Preparing your workspace
In this practical we'll be using Python/[Pandas](https://pandas.pydata.org) to explore and visualise some example data using [Jupyter Notebooks](https://jupyter.org) in [Google Colaboratory](https://colab.research.google.com).

At the start of each practical you should make a copy of the notebook in your own google drive:
- File -> Save as copy in Drive

If you would prefer to work with a Jupyter Lab session ony your own machine you can download the notebook directly via:
- File -> Downaload -> Download .ipynb

In this practial we will be preparing data for use with some data mining applications.

The data for this week can be found on [GitHub](https://github.com/PaulHancock/COMP5009_pracs).


# Heart Disease 💗 😢

The data is derived from the work of [Detrano et al. (1989)](https://pubmed.ncbi.nlm.nih.gov/2756873/), and is made available via the the UCI Machine Learning Repository ([Janosi et al (1989)](https://archive.ics.uci.edu/dataset/45/heart+disease)).

The original data set had 76 features listed, but only 14 are used in this work. They include:
1. age: age in years      
2. sex: sex (1 = male; 0 = female)      
3. cp: chest pain type:
    1. typical angina
    2. atypical angina
    3. non-anginal pain
    4. asymptomatic        
4. trestbps: resting blood pressure (in mm Hg on admission to the hospital)
5. chol: serum cholestoral in mg/dl     
6. fbs: (fasting blood sugar > 120 mg/dl)  (1 = true; 0 = false)      
7. restecg: resting electrocardiographic results
    0. normal
    1. having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV)
    2. showing probable or definite left ventricular hypertrophy by Estes' criteria  
8. thalach: maximum heart rate achieved   
9. exang: exercise induced angina (1 = yes; 0 = no)     
10. oldpeak = ST depression induced by exercise relative to rest   
11. lope: the slope of the peak exercise ST segment
    1. upsloping
    2. flat
    3. downsloping    
12. ca: number of major vessels (0-3) colored by flourosopy        
13.  thal: 3 = normal; 6 = fixed defect; 7 = reversable defect      
14. num: diagnosis of heart disease (angiographic disease status)

From the above data description we can see that this data set has a mixture of cetegorical and numerical data types, and thus it will be a good data set for us to practice calculating distances and similarity between instances.


**References:**
- Detrano R, Janosi A, Steinbrunn W, Pfisterer M, Schmid JJ, Sandhu S, Guppy KH, Lee S, Froelicher V. International application of a new probability algorithm for the diagnosis of coronary artery disease. Am J Cardiol. 1989 Aug 1;64(5):304-10. https://doi.org/10.1016/0002-9149(89)90524-9. PMID: 2756873.
- Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). Heart Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X.

### Load our dataset
This dataset is availbe via the `ucimlrepo` python package.

In [ ]:
!pip install ucimlrepo --quiet

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets
df = pd.concat([X, y], axis=1)
df.head()

As usual our dataset contains some defects that we need to deal with, thankfully we are only worried about missing data this time.

Use the skills you learned last week to find the features with missing data and action them accordinly.

In [ ]:
?

In [ ]:
# 'ca' and 'thal' are categorical and missing data so we fill missing values with the mode of the distribution
ca_mode = ?
thal_mode = ?
df.fillna(?, inplace=True)

In [ ]:
df.isnull().sum()

All the data are currently stored as some kind of numeric type (int/float), however the data description says that we have some categorical data types.

Update the data types so that we have categorical data as per the data description above.

In [ ]:
# Categorical features
categorical = [?]
df[categorical] = df[categorical].astype('category')

# Numeric features
numeric = [?]

In [ ]:
# Confirm the data type of each feature
df.dtypes

## Computing distances

Select an instance (row) of the data at random, and compute the distance between that instance and the first 10.

use the following metrics:
- L1 (Manhatten, `cityblock` according to scipy)
- L2 (Euclidean)
- L_inf (Chebyshev)

In [ ]:
my_row = ?

In [ ]:
from scipy.spatial.distance import cityblock, euclidean, chebyshev

ref = df.iloc[my_row]
top10 = df.head(10)

# Compute distances using different metrics
distances_manhattan = top10.apply(lambda row: cityblock(row, ref), axis=1)
distances_euclidean = top10.apply(lambda row: euclidean(row, ref), axis=1)
distances_chebyshev = top10.apply(lambda row: chebyshev(row, ref), axis=1)

distances = pd.DataFrame([distances_manhattan, distances_euclidean, distances_chebyshev], index=['Manhattan', 'Euclidean', 'Chebyshev']).T
distances.plot(xlabel='Instance', ylabel='Distance', title='Disance to instance 100')

What do you notice about the ordering of distances from largest to smallest for each of the instances?

Which of the 10 instances are mote like your references instance?

In [ ]:
# Inspect the most similar instance
df.iloc[?]

Look at each of the numeric features and make note of their scale (the difference between the min and the max values).

In [ ]:
df.describe()

Notice that not all the numeric features have the same scale. This means that a "distance" of 1 unit in age is equally important as a distance of 1 in ca or chol. Let us now rescale our data so that all features have the same min/max value.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[numeric] = scaler.fit_transform(df[numeric])
df_scaled[numeric].describe()

In [ ]:
ref = df_scaled.iloc[my_row]
top10 = df_scaled.head(10)

# Compute distances using different metrics
distances_manhattan = top10.apply(lambda row: cityblock(row, ref), axis=1)
distances_euclidean = top10.apply(lambda row: euclidean(row, ref), axis=1)
distances_chebyshev = top10.apply(lambda row: chebyshev(row, ref), axis=1)

distances = pd.DataFrame([distances_manhattan, distances_euclidean, distances_chebyshev], index=['Manhattan', 'Euclidean', 'Chebyshev']).T
distances.plot(xlabel='Instance', ylabel='Distance', title='Disance to instance 100')

What has changed now that we have rescaled the numeric features?

Which of the top 10 instances is now the most similar to your rference instance?

## Computing Similarities

Our data also contains categorical data so lets use these features to compare our reference row to the other instances.

sci-kit learn doesn't natively support categorical data so we'll have to home-brew a few of these similarity measures by writing python functions.

In [ ]:
def overlap_measure(row1, row2, categorical_features):
  """Calculate the overlap measure between two rows for categorical features."""
  overlap = 0
  for col in categorical_features:
    if row1[col] == row2[col]:
      overlap += 1
    # else:
    #   overlap += 0
  return overlap / len(categorical_features)

def goodall_measure(row1, row2, df, categorical_features):
  """Calculate the Goodall measure between two rows for categorical features."""
  goodall = 0
  n = len(df)
  for col in categorical_features:
    if row1[col] == row2[col]:
      p = df[col].value_counts(normalize=True)[row1[col]]  # normalised counts are equivalent to probabilities
      goodall += 1 - (p**2)
    # else:
    #   overlap += 0
  return goodall

def inverse_occurrence_frequency(row1, row2, df, categorical_features):
  """Calculates the inverse occurrence frequency similarity between two rows for categorical features."""
  iof = 0
  for col in categorical_features:
    if row1[col] == row2[col]:
      p = df[col].value_counts(normalize=True)[row1[col]]
      iof += 1 / p**2
    # else:
    #   overlap += 0
  return iof

In [ ]:
ref = df_scaled.iloc[my_row]
top10 = df_scaled.head(10)

# Compute distances using different metrics
similarity_overlap = top10.apply(lambda row: ?), axis=1)
similarity_goodall = top10.apply(lambda row: ?), axis=1)
similarity_iof     = top10.apply(lambda row: ?), axis=1)

similarities = pd.DataFrame([similarity_overlap, similarity_goodall, similarity_iof], index=['Overlap', 'Goodall', 'Inverse Occurrence Frequency']).T
similarities.plot(xlabel='Instance', ylabel='Similarity', title='Similarity with instance 100')

In [ ]:
# The index of the maximum value for each measure
similarities.idxmax()

Do all the similarity measures agree on which is the most similar instance?

Is your most similar instance the same as the smallest distance instance we computed earlier?

Since we have a mix of categorical and numeric data, we can create a similarity measure that combines all the data.

$$ SIM(x,y) = w_1 Sim(categorical) + w_2\frac{1}{1+(Dist(numerical)} $$

Where $w_1$ and $w_2$ are weighting factors that account for the different number of features being used in each of the calculations.


In [ ]:
def sim(row1, row2, df, categorical_features, numeric_features):
  """
  Computes a combined similarity measure for mixed numerical and categorical data.

  The measure is calculated as a weighted sum of Goodall similarity for categorical features
  and 1/(1+Euclidean distance) for numerical features. The weights are the proportion
  of categorical and numeric features respectively.

  Args:
    row1 (pd.Series): The first row (instance) to compare.
    row2 (pd.Series): The second row (instance) to compare.
    df (pd.DataFrame): The original DataFrame containing the data.
    categorical_features (list): A list of column names for categorical features.
    numeric_features (list): A list of column names for numeric features.

  Returns:
    float: The combined similarity score between row1 and row2.
  """
  num_categorical = len(categorical_features)
  num_numeric = len(numeric_features)
  total_features = num_categorical + num_numeric

  # Calculate Goodall similarity for categorical features
  goodall_sim = ?
  weight_categorical = num_categorical / total_features

  # Ensure rows have only numeric features for Euclidean distance
  euclidean_dist = ?
  # Avoid division by zero if euclidean_dist is very close to 0
  if euclidean_dist < 1e-9:
      numeric_sim = 1e-9
  else:
      numeric_sim = 1 / (1 + euclidean_dist)
  weight_numeric = num_numeric / total_features

  # Combine the similarities with relative weights
  combined_similarity = (weight_categorical * goodall_sim) + (weight_numeric * numeric_sim)

  return combined_similarity

# Example usage of the sim function with the loaded data
ref = df_scaled.iloc[my_row]
top10 = df_scaled.head(10)

combined_similarities = top10.apply(lambda row: sim(row, ref, df_scaled, categorical, numeric), axis=1)

print("\nCombined Similarities with instance 100:")
combined_similarities.plot(xlabel='Instance', ylabel='Similarity', title='Combined Similarity with instance 100')
print(f"The most similar instances is instance {combined_similarities.idxmax()}")


Now that we can compute the distance between any two data points, lets compute the distnaces between our reference row and all other data points, looking for the top 5 most similar.

In [ ]:
df_scaled.apply(lambda row: sim(?, axis=1).sort_values(ascending=False).head()


Take this one step further and compute the distance between all pairs of data points. Since the full NxN will take a while to compute we'll instead limit this to just the pairs within the first 10 instances. This distance/similarity matrix is something that we'll come back to in a later practical when we start looking at clustering.

In [ ]:
import numpy as np
import itertools

# Select the first 10 entries from the scaled DataFrame
df_subset = df_scaled.head(10)

# A 10x10 array to store our similarities, default values are empty (nan)
sims = np.zeros((10, 10)) * np.nan

# fill in the upper triangular part of the matrix
for i in range(10):
  for j in range(i+1,10):
    sims[i,j] = ?

# Convert the square distance matrix to a pandas DataFrame for better readability
distance_df = pd.DataFrame(sims, index=df_subset.index, columns=df_subset.index)

print("\nCombined Distance Matrix for the first 10 entries:")
distance_df

Reflect on the following questions:
- Our data were initial all numeric, however we changed some to categorical after reading the data description. How does this kind of feature encoding influence distance/similarity measures?
- How did scaling our data set change our view of which instances (rows) were close to each other?
- We used the goodall measure and the euclidean distance when creating our combined similarity measure. What are some of the other options available to us, and how might they have affected our final results?
- How might data cleaning errors (e.g., missing values, inconsistent types) affect distance calculations?


Summarise the work that you have done today. For each step describe the problem being faced, the solution that was chosen, and the result of your actions.